## Download the exercise data
Run the next cell once before starting the exercise. It downloads and extracts this notebook’s data into `~/kenya2026`. Set `KENYA2026_WORK_DIR` first if you prefer another location.

In [ ]:
from pathlib import Path
import os
import subprocess

exercise = "day2_afternoon_psmc"
base_url = "https://popgen.dk/albrecht/course/kenya2026/data"
work_dir = Path(os.environ.get("KENYA2026_WORK_DIR", Path.home() / "kenya2026")).expanduser()
exercise_dir = work_dir / exercise
archive = work_dir / f"{exercise}.zip"
work_dir.mkdir(parents=True, exist_ok=True)

if not archive.exists():
    subprocess.run(["wget", "-c", f"{base_url}/{exercise}.zip", "-O", str(archive)], check=True)
if not exercise_dir.exists():
    subprocess.run(["unzip", "-q", str(archive), "-d", str(work_dir)], check=True)

os.chdir(exercise_dir)
print(f"Working directory: {Path.cwd()}")

### Software requirements
The setup cell above downloads **only the exercise data**. It does not install software. Before running the rest of this notebook, install the command-line programs and the Python or R packages that are imported or called in the exercises. If you see an error such as `command not found`, `ModuleNotFoundError`, or `there is no package called ...`, install the named dependency or ask an instructor for help.

# 1. Resources and setup
- [`PSMC`](https://github.com/lh3/psmc)
- [`VCF file format`](https://samtools.github.io/hts-specs/VCFv4.2.pdf)
- [`vcftools`](https://vcftools.github.io/index.html)
- [`1000 Genomes high-coverage variant calls`](https://www.internationalgenome.org/data-portal/sample)


Let us start by defining some variables and setting up the environment before we run the analyses.

In [ ]:
## Set up location for the software
TOOLS_PATH=/course/popgen25/software
SIM_DATA_PATH=sim_data
REAL_DATA_PATH=real_data
SCRIPTS_PATH=/course/popgen25/demography/scripts

## Set up variables for executables
PSMC=${TOOLS_PATH}/psmc/psmc
VCF2PSMCFA=${SCRIPTS_PATH}/vcf2psmcfa.py
PSMC_PLOT=${TOOLS_PATH}/psmc/utils/psmc_plot.pl

Now, let us set up the directories for running the demography analyses.

In [ ]:
## Make directory
cd ~
mkdir -p demography
cd demography

In [ ]:
import os
os.chdir("demography")

For convenience, this tutorial is presented as a Python notebook. Almost all of its commands are run in Bash.

We will use `psmc` to estimate effective population size from our data. Rather than starting directly from BAM files, this exercise uses a Variant Call Format (VCF) file. We will first inspect a VCF, then convert it into PSMC input with a custom Python script, and finally run PSMC on simulated and real data. PSMC relies on high-coverage data from which genotypes can be estimated accurately, which is one reason we use prepared VCF files in this exercise. 

# 2. Data
In this tutorial, we will use two datasets. First, we will use samples from three simulated populations: one with a constant population size, one with population growth, and one with population decline (see the figure later in the tutorial). In the second half of the exercise, we will use three individuals from the wildebeest project: one black wildebeest and two blue wildebeest from Tanzania and Kenya. We will compare how their inferred effective population-size histories vary. You can also use these scripts as a starting point for exploring other datasets. 

Before we begin, let us look at the VCF format using the simulated data, which contain three individuals, one from each simulated population. The VCF files are compressed, so we will use `zcat` to display one.

In [ ]:
zcat ${SIM_DATA_PATH}/threeinds.recode.vcf.gz | head -10

Lines beginning with `##` are metadata headers that describe the VCF. The line beginning with `#CHROM` labels the columns in the subsequent data lines. Each data line contains information about a single variant. Each individual, s1_1, s2_1, and s3_1, has a column containing its genotype at that variant. Here, "0|0" is homozygous for the reference (REF) allele, "0|1" is heterozygous, and "1|1" is homozygous for the alternate (ALT) allele.

We will use the VCF files to generate input for our PSMC analysis. Note that this is __not__ the preferred method for preparing PSMC input. The standard workflow begins with BAM files and creates a filtered diploid consensus sequence; details are available [here](https://github.com/lh3/psmc). 

__Question:__ Why do you think VCF files are not the preferred format? What information might be missing?

# 3. Working with simulated data

In this first exercise, we will look at simulated data from demography that looks like this.

In [ ]:
## We are using python to see the images - we will do the same later for our PSMC results.
from matplotlib import pyplot as plt
from matplotlib import image as mpimg
fig, ax = plt.subplots(figsize=(5, 10))
ax.axis('off')
image = mpimg.imread("/course/popgen25/demography/images/popsize.png")
ax.imshow(image)

We will use PSMC to reconstruct the simulated population-size histories and compare the results with the known histories. First, let us construct a PSMCFA file, the FASTA-like input format used by PSMC, from the VCF. 

## PSMCFA file

A PSMCFA file is a FASTA-like representation of the genome, or part of a genome, in which each letter indicates whether a fixed-size window contains a heterozygous variant. We will create one with the custom script `vcf2psmcfa.py`. This script expects the VCF to contain only one chromosome or scaffold, which is not generally true of a VCF. It takes two input parameters: the VCF filename and the sample name. The output filename is derived from the sample name, and the script uses a fixed window size of 100 bp. 

In [ ]:
$VCF2PSMCFA ${SIM_DATA_PATH}/threeinds.recode.vcf.gz s1_1
$VCF2PSMCFA ${SIM_DATA_PATH}/threeinds.recode.vcf.gz s2_1
$VCF2PSMCFA ${SIM_DATA_PATH}/threeinds.recode.vcf.gz s3_1

Let us take a quick look at one of the PSMCFA files. 

In [ ]:
head s2_1.psmcfa

It appears to be a regular FASTA file containing only "T" and "K". Here, "T" represents a 100 bp window without a heterozygous site, whereas "K" represents a 100 bp window with at least one heterozygous site. 

__Question:__ Given the population histories, which of the three samples would you expect to have the highest number of heterozygous windows?

### Quick check: PSMC input and heterozygosity

Run the following cell for a short quiz before moving on to the PSMC model.

In [ ]:
from jupyterquiz import display_quiz

display_quiz('/davidData/users/thomas/workshop/psmc_quizzes/psmc_quiz1_input_and_heterozygosity.json')

## Running PSMC

Now it is time to run our first PSMC analysis. Let us begin by looking at the available options. 

In [ ]:
$PSMC

In particular, note the `-p` option, which specifies how PSMC divides time into atomic intervals and groups adjacent intervals that share an estimated population-size parameter. Read the pattern from left to right:

- A number by itself means that many consecutive atomic intervals share one parameter.
- `A*B` means `A` consecutive groups of `B` atomic intervals, with one parameter estimated for each group.

For example, the coarse pattern `4+5*3+4` is decoded as follows:

| Term | Atomic intervals | Free parameters |
|---|---:|---:|
| `4` | 4 | 1 |
| `5*3` | 15 | 5 |
| `4` | 4 | 1 |
| **Total** | **23** | **7** |

The parameter pattern controls model flexibility. More parameter groups can represent finer changes through time, but each parameter is then supported by less information and may become noisy or unstable. Grouping intervals sacrifices some temporal resolution in exchange for more stable estimates, especially in the most recent and oldest parts of the history.

Let us now run PSMC for the first time. We will use the default values for the other options while explicitly specifying the parameter pattern. This pattern is quite coarse, but we will repeat the analysis with finer time intervals in the next section.

__Question:__ Why are the most recent and oldest time intervals often grouped so that several intervals share one parameter? You may be able to answer this more fully after seeing the output.

In [ ]:
$PSMC -p "4+5*3+4" -o s1_1_coarsePattern.psmc s1_1.psmcfa
$PSMC -p "4+5*3+4" -o s2_1_coarsePattern.psmc s2_1.psmcfa
$PSMC -p "4+5*3+4" -o s3_1_coarsePattern.psmc s3_1.psmcfa

Let us take a look at one of the output files, and discuss its contents. 

In [ ]:
tail -34 s1_1_coarsePattern.psmc

### Reading the PSMC output

The output contains several record types. For this exercise, focus on the following:

- `RD` gives the optimization iteration.
- `LK` gives the log likelihood at that iteration; its stabilization across later iterations is one useful convergence check.
- `TR` reports the fitted scaled mutation and recombination parameters.
- Each `RS` line describes one atomic time interval. In `RS k t_k lambda_k ...`, `k` is the interval index, `t_k` is scaled time, and `lambda_k` is population size relative to the fitted reference size.
- `PA` provides a compact record of the parameter pattern and fitted parameter values.

Notice that consecutive `RS` lines can have the same `lambda_k`. Those intervals were tied together by the `-p` pattern and therefore share one population-size parameter.

Let us now concatenate the PSMC outputs for all three samples and plot them in one PDF. 

In [ ]:
cat s1_1_coarsePattern.psmc s2_1_coarsePattern.psmc s3_1_coarsePattern.psmc > combined_coarsePattern.psmc 

We will use the built-in PSMC plotting script to plot them. 

In [ ]:
$PSMC_PLOT

### Scaling and plotting assumptions

PSMC estimates population size and time in scaled units. The plotting script converts them into effective population size and calendar time using assumptions supplied by the user:

- `-u` is the mutation rate per nucleotide per generation. It scales both inferred $N_e$ and time in generations.
- `-g` is the number of years per generation. It changes the time axis in years but does not change inferred $N_e$.
- `-s` is the PSMCFA window size and must match the value used during input preparation; here it is 100 bp.
- `-n 30` selects the estimates from iteration 30.
- `-Y` changes only the displayed upper limit of the population-size axis; it does not change the fitted model.

Changing `-u` or `-g` rescales the plotted axes; it does not rerun or refit PSMC. For simulated data, the scaling values should match those used to generate the simulation. The wildebeest plots later use `-u 1.45e-08` and `-g 7.5`, following the assumptions in the [wildebeest genome study](https://doi.org/10.1038/s41467-024-47015-y). Mutation rates and generation times are calibration assumptions rather than universal constants.

The simulation command below sets `-u 2.5e-08` but does not specify `-g`, so the plotting script uses its default of 25 years per generation.

In [ ]:
$PSMC_PLOT -u 2.5e-08 -s 100 -Y 1 -m 5 -n 30 -p -M "Pop1, Pop2, Pop3" Simulations_coarsePattern combined_coarsePattern.psmc

Let us take a look at the plot. First, we will convert the PDF to a PNG for easier viewing in Python.

In [ ]:
# Convert the PDF to a PNG. pdftoppm adds -1.png to the output filename.
pdftoppm -png Simulations_coarsePattern.pdf Simulations_coarsePattern

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
plt.axis('off')
image = mpimg.imread("Simulations_coarsePattern-1.png")
plt.imshow(image, aspect="auto")

__Question:__ Can you see the differences in the population sizes? Do they make sense to you? Do they match up with the simulation scheme?

Now that we know how to run PSMC and plot its output, repeat the analysis for the same samples using the finer time pattern `4+25*2+4+6`, and plot the results. 

__Question:__ What would you expect the output to be? 

__Question:__ Does the output plot from the finer time pattern match the coarser time pattern? Which one would you prefer?

__Question:__ When would you not always use a finer time partition?

__Bonus task:__ A recent [article](https://www.biorxiv.org/content/10.1101/2024.06.17.599025v1) investigated how time-partitioning parameters affect the signal of recent expansion in PSMC analyses. Explore whether different parameter patterns help. First, keep the total number of atomic intervals fixed while changing their grouping; then change the total number of intervals. Example patterns include `2+10*2+1`, `6+11*1+6`, `4+20*2+6`, and `2+30*1+2`. Compare the resulting curves, convergence, and output logs.


In [ ]:
## Your turn: repeat the simulated analysis using the finer time pattern.
## Run PSMC for all three samples, combine the outputs, plot the curves,
## and convert the PDF to a PNG.

# Write your commands below:



### Quick check: time partitioning

Run the following cell to check your understanding of PSMC time intervals and parameter resolution.

In [ ]:
from jupyterquiz import display_quiz

display_quiz('/davidData/users/thomas/workshop/psmc_quizzes/psmc_quiz2_time_partitioning.json')

# 4. PSMC on real wildebeest samples

We will now estimate effective population-size histories from three wildebeest samples. We have already generated the PSMCFA files using our custom `vcf2psmcfa.py` script. 

__Task:__ The PSMCFA files are called CGnoNa_58015_chr27.psmcfa, CTauTzS_3709_chr27.psmcfa, and CTauKeS__698_chr27.psmcfa. They are in `$REAL_DATA_PATH`. We extracted data from chromosome 27 only to reduce runtime. Using the same commands as in the previous section, run PSMC on these three samples with the pattern `4+25*2+4+6`. This will take several minutes.

In [ ]:
## PSMC takes several minutes to run. If you are short on time, copy the precomputed results.
## Otherwise, comment out the cp command and uncomment the PSMC commands below.
cp $REAL_DATA_PATH/*chr27_finePattern.psmc .

### Uncomment these commands to run PSMC.
# $PSMC -p "4+25*2+4+6" -o blackNamibia_58015_chr27_finePattern.psmc ${REAL_DATA_PATH}/CGnoNa_58015_chr27.psmcfa &
# $PSMC -p "4+25*2+4+6" -o blueTanzania_3709_chr27_finePattern.psmc ${REAL_DATA_PATH}/CTauTzS_3709_chr27.psmcfa &
# $PSMC -p "4+25*2+4+6" -o blueKenya_698_chr27_finePattern.psmc ${REAL_DATA_PATH}/CTauKeS__698_chr27.psmcfa &
# wait

echo "# Output files from PSMC"
ls *chr27_finePattern.psmc


Let's plot the results. First, we combine the three files. The plotting commands then create a file called combined_wilde_finePattern-1.png.

In [ ]:
## Combine the outputs into one file.
cat blackNamibia_58015_chr27_finePattern.psmc blueTanzania_3709_chr27_finePattern.psmc blueKenya_698_chr27_finePattern.psmc > combined_wilde_finePattern.psmc

## Plot the results.
$PSMC_PLOT -u 1.45e-08 -g 7.5 -Y 10 -s 100 -m 5 -n 30 -p -M "blackNamibia, blueTanzania, blueKenya" combined_wilde_finePattern combined_wilde_finePattern.psmc
pdftoppm -png combined_wilde_finePattern.pdf combined_wilde_finePattern

View the plot with this command.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
plt.axis('off')
image = mpimg.imread("combined_wilde_finePattern-1.png")
plt.imshow(image)

__Question:__ Can you guess which specific area of Tanzania sample CTauTzS_3709 is from? To which blue wildebeest subspecies might it belong? Hint: look at the results presented in our wildebeest genome paper.

__Question:__ Interpret these effective population size plots for the three wildebeest populations. 

To explore higher resolution towards the present time, run the following analysis for the black wildebeest sample. This unmasked run also provides the baseline for the low-coverage comparison in the next section. Reflect on what changes and why.

In [ ]:
$PSMC -p "1+1+1+1+25*2+4+6" -o blackNamibia_58015_chr27_fineRecentPattern.psmc ${REAL_DATA_PATH}/CGnoNa_58015_chr27.psmcfa
$PSMC_PLOT -u 1.45e-08 -g 7.5 -Y 10 -s 100 -m 5 -n 30 -p blackNamibia_58015_chr27_fineRecentPattern blackNamibia_58015_chr27_fineRecentPattern.psmc
pdftoppm -png blackNamibia_58015_chr27_fineRecentPattern.pdf blackNamibia_58015_chr27_fineRecentPattern



Here is the resulting plot.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
plt.axis('off')
image = mpimg.imread("blackNamibia_58015_chr27_fineRecentPattern-1.png")
plt.imshow(image)

## Effect of low coverage

In this section, we will explore how low coverage affects effective population-size estimation. First, consider the following question.

__Question:__ What happens to genotype calling as coverage decreases for a single individual? Which genotypes become harder to call accurately: homozygous or heterozygous genotypes?

We have prepared an input file for sample CGnoNa_58015 in which we intentionally masked 20% of the heterozygous sites to mimic the effects of low coverage. This file, called CGnoNa_58015_chr27_missing_20pctHets.psmcfa, is in `$REAL_DATA_PATH`. We will run PSMC on it and plot the result together with the unmasked run from the previous section, so make sure you run the preceding PSMC cell first.

__Question:__ What do you expect will happen to the effective population size estimates?

In [ ]:
$PSMC -p "1+1+1+1+25*2+4+6" -o blackNamibia_58015_chr27_missing_20pctHets.psmc ${REAL_DATA_PATH}/CGnoNa_58015_chr27_missing_20pctHets.psmcfa

cat blackNamibia_58015_chr27_fineRecentPattern.psmc blackNamibia_58015_chr27_missing_20pctHets.psmc > combined_blackNamibia_chr27_coverageComparison.psmc

$PSMC_PLOT -u 1.45e-08 -g 7.5 -Y 10 -s 100 -m 5 -n 30 -p -M "unmasked, 20% heterozygotes masked" blackNamibia_chr27_coverageComparison combined_blackNamibia_chr27_coverageComparison.psmc
pdftoppm -png blackNamibia_chr27_coverageComparison.pdf blackNamibia_chr27_coverageComparison



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
plt.axis('off')
image = mpimg.imread("blackNamibia_chr27_coverageComparison-1.png")
plt.imshow(image)

__Question:__ Why does masking heterozygous sites shift the inferred $N_e$ curve downward?

## Estimating uncertainty in $N_e$ estimates

Due to time constraints, we will not examine the procedure in detail. PSMC uncertainty can be estimated by splitting the genome into chunks and resampling those chunks in bootstrap replicates. Here is an example of bootstrapped estimates: 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
plt.axis('off')
image = mpimg.imread("/course/popgen25/demography/images/bootstrap.png")
plt.imshow(image)

__Question:__ Why do the most recent time periods have the highest variance in $N_e$ estimates?

### Quick check: interpretation and uncertainty

Run the following cell for a final quiz on interpreting PSMC results and their limitations.

In [ ]:
from jupyterquiz import display_quiz

display_quiz('/davidData/users/thomas/workshop/psmc_quizzes/psmc_quiz3_interpretation_and_uncertainty.json')